# app_2_resilience_analysis Combined
This notebook combines all scripts from `app_2_resilience_analysis`.

### future_weather_forecast.py

In [ ]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import os

# Set up the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Target destinations based on user requirement
DESTINATIONS = {
    'Andalucía': {'lat': 37.38, 'lon': -5.98},
    'Rome': {'lat': 41.90, 'lon': 12.49},
    'Madrid': {'lat': 40.41, 'lon': -3.70},
    'Islas Baleares': {'lat': 39.69, 'lon': 3.01},
    'Albania': {'lat': 41.15, 'lon': 20.16},
    'Nord Jylland': {'lat': 57.04, 'lon': 9.91},
    'Region Syd': {'lat': 55.39, 'lon': 9.38}
}

def fetch_future_forecast(out_dir="."):
    os.makedirs(out_dir, exist_ok=True)
    all_destinations_data = []

    for name, coords in DESTINATIONS.items():
        print(f"Fetching 14-day weather forecast for {name}...")
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": coords['lat'],
            "longitude": coords['lon'],
            "daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"],
            "timezone": "auto",
            "forecast_days": 14
        }
        
        try:
            responses = openmeteo.weather_api(url, params=params)
            response = responses[0]
            
            # Process daily data
            daily = response.Daily()
            daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
            daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
            daily_precipitation_sum = daily.Variables(2).ValuesAsNumpy()
            
            daily_data = {"date": pd.date_range(
                start=pd.to_datetime(daily.Time(), unit="s", utc=True),
                end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=daily.Interval()),
                inclusive="left"
            )}
            daily_data["temp_max"] = daily_temperature_2m_max
            daily_data["temp_min"] = daily_temperature_2m_min
            daily_data["precipitation_sum"] = daily_precipitation_sum
            
            daily_dataframe = pd.DataFrame(data=daily_data)
            daily_dataframe['destination'] = name
            
            all_destinations_data.append(daily_dataframe)
            
        except Exception as e:
            print(f"Failed to fetch forecast for {name}: {e}")

    if all_destinations_data:
        final_df = pd.concat(all_destinations_data, ignore_index=True)
        # We can just keep the daily forecast natively since it's only 14 days
        final_df['date'] = final_df['date'].dt.date
        out_path = os.path.join(out_dir, "future_forecast_14_days.csv")
        final_df.to_csv(out_path, index=False)
        print(f"Future forecast aggregated and saved to {out_path}")
    else:
        print("No forecast data was extracted.")

if __name__ == "__main__":
    fetch_future_forecast(out_dir="app_2_resilience_analysis/output")


### weather_resilience_index.py

In [ ]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import os

# Set up the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Target destinations based on user requirement
DESTINATIONS = {
    'Andalucía': {'lat': 37.38, 'lon': -5.98},
    'Rome': {'lat': 41.90, 'lon': 12.49},
    'Madrid': {'lat': 40.41, 'lon': -3.70},
    'Islas Baleares': {'lat': 39.69, 'lon': 3.01},
    'Albania': {'lat': 41.15, 'lon': 20.16},
    'Nord Jylland': {'lat': 57.04, 'lon': 9.91},
    'Region Syd': {'lat': 55.39, 'lon': 9.38}
}

def fetch_historical_weather(start_date="2020-01-01", end_date="2024-12-31", out_dir="."):
    os.makedirs(out_dir, exist_ok=True)
    all_destinations_data = []

    for name, coords in DESTINATIONS.items():
        print(f"Fetching historical weather for {name}...")
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude": coords['lat'],
            "longitude": coords['lon'],
            "start_date": start_date,
            "end_date": end_date,
            "daily": ["temperature_2m_mean", "temperature_2m_max", "relative_humidity_2m_mean"],
            "timezone": "auto"
        }
        
        try:
            responses = openmeteo.weather_api(url, params=params)
            response = responses[0]
            
            # Process daily data
            daily = response.Daily()
            daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
            daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
            daily_relative_humidity_mean = daily.Variables(2).ValuesAsNumpy()
            
            daily_data = {"date": pd.date_range(
                start=pd.to_datetime(daily.Time(), unit="s", utc=True),
                end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=daily.Interval()),
                inclusive="left"
            )}
            daily_data["temp_mean"] = daily_temperature_2m_mean
            daily_data["temp_max"] = daily_temperature_2m_max
            daily_data["humidity_mean"] = daily_relative_humidity_mean
            
            daily_dataframe = pd.DataFrame(data=daily_data)
            
            # Add Month column
            daily_dataframe['month'] = daily_dataframe['date'].dt.to_period('M')
            
            # Identify Heatwave Days (e.g., max temp > 30C)
            daily_dataframe['is_heatwave'] = daily_dataframe['temp_max'] > 30.0
            
            # Aggregate to Monthly
            monthly_agg = daily_dataframe.groupby('month').agg(
                avg_temp_mean=('temp_mean', 'mean'),
                avg_temp_max=('temp_max', 'mean'),
                avg_humidity=('humidity_mean', 'mean'),
                heatwave_days=('is_heatwave', 'sum')
            ).reset_index()
            
            monthly_agg['destination'] = name
            all_destinations_data.append(monthly_agg)
            
        except Exception as e:
            print(f"Failed to fetch data for {name}: {e}")

    if all_destinations_data:
        final_df = pd.concat(all_destinations_data, ignore_index=True)
        final_df['month'] = final_df['month'].astype(str)
        out_path = os.path.join(out_dir, "historical_weather_resilience.csv")
        final_df.to_csv(out_path, index=False)
        print(f"Historical weather aggregated and saved to {out_path}")
    else:
        print("No historical data was extracted.")

if __name__ == "__main__":
    fetch_historical_weather(out_dir="app_2_resilience_analysis/output")


### wiki_tourist_ratio.py

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import os

def calculate_ratio_and_classify(merged_dataset, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    print(f"Loading {merged_dataset} for cultural analysis...")
    
    if not os.path.exists(merged_dataset):
        print("Merged dataset not found. Please run Merger app first, or provide valid path.")
        return
        
    df = pd.read_csv(merged_dataset)
    
    # We require total_visitors and total_wiki_views
    # Assuming the merge_destinations script mapped 'OBS_VALUE' or something similar to 'total_visitors'
    vis_col = [c for c in df.columns if 'visitor' in c.lower() or 'obs' in c.lower()]
    
    if not vis_col or 'total_wiki_views' not in df.columns:
        print("Dataset missing required columns for wiki_tourist_ratio calculation.")
        print(f"Found columns: {df.columns.tolist()}")
        return
        
    actual_vis_col = vis_col[-1] # Usually OBS_VALUE etc
    df['total_visitors'] = pd.to_numeric(df[actual_vis_col], errors='coerce')
    df['total_wiki_views'] = pd.to_numeric(df['total_wiki_views'], errors='coerce')
    
    # Remove rows where visitors is 0 or NaN to prevent division by zero
    valid_df = df.dropna(subset=['total_visitors', 'total_wiki_views'])
    valid_df = valid_df[valid_df['total_visitors'] > 0]
    
    # Calculate fraction
    valid_df['wiki_tourist_fraction'] = valid_df['total_wiki_views'] / valid_df['total_visitors']
    
    # Machine Learning - KMeans Classification
    # Features: wiki_tourist_fraction, and perhaps normalized total_visitors
    features = valid_df[['wiki_tourist_fraction', 'total_visitors']].copy()
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)
    
    # Try 2 clusters: Cultural vs. Weather/Mass Tourism
    kmeans = KMeans(n_clusters=2, random_state=42)
    valid_df['tourism_class'] = kmeans.fit_predict(scaled_features)
    
    # Identify which class is "Cultural"
    # Usually cultural class has higher wiki views per tourist and potentially less MASSive seasonality spikes
    cluster_means = valid_df.groupby('tourism_class')['wiki_tourist_fraction'].mean()
    cultural_class_id = cluster_means.idxmax()
    
    valid_df['classification_label'] = valid_df['tourism_class'].apply(
        lambda x: 'Cultural-based' if x == cultural_class_id else 'Weather-based / Mass'
    )
    
    # Plotting the classification
    plt.figure(figsize=(10, 6))
    
    cultural_pts = valid_df[valid_df['tourism_class'] == cultural_class_id]
    weather_pts = valid_df[valid_df['tourism_class'] != cultural_class_id]
    
    plt.scatter(weather_pts['total_visitors'], weather_pts['wiki_tourist_fraction'], 
                alpha=0.5, label='Weather-based / Mass', color='orange')
    plt.scatter(cultural_pts['total_visitors'], cultural_pts['wiki_tourist_fraction'], 
                alpha=0.5, label='Cultural-based', color='blue')
        
    plt.xlabel('Total Visitors')
    plt.ylabel('Wiki Views / Tourist Fraction')
    plt.title('Tourism Classification using K-Means (Wiki Fraction vs Visitors)')
    plt.legend()
    plt.yscale('log')
    plt.xscale('log')
    
    plot_path = os.path.join(out_dir, "tourism_classification_plot.png")
    plt.savefig(plot_path)
    print(f"Classification plot saved to {plot_path}")
    
    # Save the output CSV
    out_csv = os.path.join(out_dir, "cultural_classified_destinations.csv")
    valid_df.to_csv(out_csv, index=False)
    print(f"Data classified and saved to {out_csv}")

if __name__ == "__main__":
    # We use the dataset produced by app 1 or the base dataset if testing
    merged_data_path = "app_1_api_merger/all_destinations.csv"
    calculate_ratio_and_classify(merged_data_path, "app_2_resilience_analysis/output")
